Python from scratch
===================

**Author:** Ethan Ligon



## Who this is for



You have not written much Python, or any.  That is the expected starting
point for this workshop and nothing in the sessions assumes otherwise.

What follows is not a programming course.  It is the smallest set of ideas
that lets you read the workshop notebooks and change them without guessing
— about twenty minutes, and it runs on the workshop server as it stands.

You are not starting from nothing.  You know what a variable is, what a
mean is, what a table of data looks like and what you want to do to one.
The gap is only notation.  This tutorial is about the notation.

-   **If you have used Stata:** read `python_from_stata.org` instead.  It
    covers exactly the same ground, but says at each step how the idea maps
    onto something you already do, which is faster if the mapping exists.



## Setup



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import lsms_library as ll

print("ready")

Python by itself is a general language that knows nothing about surveys or
statistics.  `import` brings in a library that does.  `pandas` handles
tables of data; `lsms_library` is the workshop's own, for household
surveys.  The `as pd` part just gives it a shorter name, because you will
type it often.

Running a cell that ends in `print("ready")` and seeing `ready` means the
libraries loaded.  If it fails instead, that is worth reporting rather than
working around.



### What code will print "Hello World"?



-   **Ask:** How do I make Python display a message?
-   **Answer:** `print("Hello World")`.  The quotation marks mark the text as
    text; without them Python would look for something *named* Hello and
    fail.  `print` is the general way to make a cell show you something, and
    you can put several things in it: `print("rows:", 246594)`.



## 1.  Names, values, and types



A name refers to a value.  The equals sign does not assert that two things
are equal — it puts a value into a name.



In [1]:
region = "Ashanti"
households = 1735
mean_size = 3.28

print(type(region), type(households), type(mean_size))
print(region.upper(), households * 2, round(mean_size, 1))

Three kinds of value, and Python calls them `str` (text), `int` (whole
number) and `float` (number with a decimal part).

The type is not bookkeeping; it decides what is legal.  `households * 2` is
3470, but `region * 2` is `"AshantiAshanti"`, because multiplying text
repeats it.  Asking for something a type cannot do is one of the most
common errors you will see:



### What does TypeError mean?



-   **Ask:** I got `TypeError: can only concatenate str (not "int") to str`.
-   **Answer:** You tried to combine a piece of text with a number, most often
    with `+`.  Python will not guess which you meant.  Either turn the number
    into text — `("total: " + str(1735))` — or, better, let `print` do it
    for you: `print("total:", 1735)`.  `TypeError` always means "this
    operation is not defined for these kinds of value".



## 1.  Holding more than one thing



A single name holding a single number is rarely enough.  Python has three
containers you will meet constantly.



In [1]:
regions = ["Ashanti", "Northern", "Volta"]        # list: ordered, editable
weights = {"Ashanti": 1.84, "Northern": 0.65}     # dict: look up by name
key = ("2016-17", "Ashanti")                      # tuple: fixed

print(regions[0], "of", len(regions))
print(weights["Ashanti"])
print(key, type(key))

-   **List:** several values in order, written in square brackets.  You reach
    one by position, and **positions start at zero**, so `regions[0]` is the
    first.  This is the single most common off-by-one mistake, and everyone
    makes it once.
-   **Dict:** pairs of a label and a value, written in curly brackets.  You
    reach a value by its label rather than its position, which is what you
    want whenever the label is meaningful.
-   **Tuple:** like a list but fixed once made, written in round brackets.  It
    looks like a minor variation and you can treat it as one until section 5,
    where it turns out that the label identifying a row of survey data is a
    tuple.



## 1.  Objects, and what the full stops are doing



Every value in Python carries its own set of things it can do.  You reach
them with a full stop: the name of the value, then a dot, then the thing
you want.



In [1]:
name = "Upper West"

print(name.lower())        # a method: something the text can do
print(len(name))           # a function: stands on its own

counts = [3, 1, 2]
counts.sort()              # a method that CHANGES counts
print(counts)

`name.lower()` reads as "take `name`, and ask it for its lowercase form".
The brackets mean "do it now".  `len(name)` is the older style, a function
that takes the value as an argument, and both exist — you will use each
without having to decide why.

Because what comes back is itself a value with its own methods, they chain.
`df.groupby("t").size()` is three steps read left to right: take `df`,
group it, count each group.  Long chains are normal and are read the same
way, one step at a time.



### Why is there a dot in the middle of the code?



-   **Ask:** What does the full stop mean in `s.groupby("v")["weight"].mean()`?
-   **Answer:** Each dot asks the thing on its left for something.  Read left to
    right: take `s`, group it by `v`, take the `weight` column of the result,
    then average it.  Every step hands its answer to the next.  If a long
    chain is hard to follow, break it up — give the middle results names and
    take one line at a time.  The result is identical.

Two kinds of thing hang off a dot, and telling them apart saves confusion:



In [1]:
small = pd.DataFrame({"region": ["Ashanti", "Northern"],
                      "weight": [1.84, 0.65]})

print(small.shape)         # attribute: a fact, no brackets
print(small.head(1))       # method: an action, brackets

-   **Attribute:** a fact the value already knows, like `.shape`, its number of
    rows and columns.  No brackets.
-   **Method:** an action it performs when asked, like `.head()`.  Brackets,
    even when there is nothing to put inside them.

Writing `small.shape()` gives `TypeError: 'tuple' object is not callable`,
which is Python saying you asked it to *do* something that was only a fact.

That `pd.DataFrame` is the other thing worth noticing here: a table, built
from a dict whose labels became column names.  It is the object the whole
workshop revolves around.



## 1.  Real data



The workshop server keeps prepared survey extracts in your home directory.
`ll.tools.get_dataframe` reads one into a DataFrame.



In [1]:
path = os.path.expanduser("~/extracts/GhanaLSS_household_roster.parquet")
roster = ll.tools.get_dataframe(path)

print("rows and columns:", roster.shape)
print("index names     :", roster.index.names)
roster.head(3)

246,594 rows and eight columns: every person in every round of the Ghana
Living Standards Survey.  `.head(3)` shows the first three so you can see
the shape of it without printing a quarter of a million lines.



## 1.  The index



Look again at what that printed.  Eight columns — `Sex`, `Age` and so on
— but also four *index names*: `i`, `t`, `v`, `pid`.

A DataFrame separates two different kinds of information.  The **columns**
are what was measured.  The **index** is the labelling that says which row
this is: here household, wave, cluster and person.  Every row therefore has
a label made of four values together — which is a tuple, and the reason
section 2 mentioned them.

This separation is useful once you are used to it and confusing before
then, because a name in the index is not a column and will not be found as
one.



### Why does my column not exist?



-   **Ask:** `roster["t"]` gives `KeyError: 't'`, but I can see `t` in the
    output.
-   **Answer:** `t` is an index level, not a column, so looking for it among
    the columns fails.  `KeyError` always means "I looked for that label and
    it is not there".  The simplest fix while you are learning is
    `roster.reset_index()`, which turns every index level into an ordinary
    column and hands you a plain rectangle.



In [1]:
flat = roster.reset_index()

print(flat.columns.tolist())
print("still", flat.shape[0], "rows")

Twelve columns now rather than eight: the four index levels became ordinary
columns, and no rows were lost.  You will see `reset_index()` throughout
the workshop notebooks, and this is all it is doing.



## 1.  Choosing rows



To work with some of the rows, build a column of true and false and use it
to select.



In [1]:
w7 = flat[flat["t"].astype(str) == "2016-17"]
print("2016-17 people:", len(w7))

adults = w7[w7["Age"] >= 18]
print("aged 18 or over:", len(adults))
print("mean age       :", round(float(w7["Age"].mean()), 1))

Read `flat["t"].astype(str) =` "2016-17"= as a question asked of every row
at once, giving true or false for each; the outer brackets then keep the
true ones.  Note the doubled equals sign: a single `=` puts a value into a name,
while `==` asks whether two things are the same.

59,864 people in the 2016–17 round, 31,954 of them adults, mean age 25.0.

The important part is what `w7` is: a **new** table.  Selecting rows did not
remove anything from `flat`, which is unchanged and still has all 246,594
rows.  Almost everything in pandas works this way — it hands you a new
result rather than altering what you started with — and that is behind
the one mistake worth learning before you make it.



## 1.  Changing values, and the trap



In [1]:
toy = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

toy[toy["a"] > 1]["b"] = 0            # looks right, does nothing
print("after the obvious version:", toy["b"].tolist())

toy.loc[toy["a"] > 1, "b"] = 0        # what to write instead
print("after .loc            :", toy["b"].tolist())

`[10, 20, 30]`, then `[10, 0, 0]`.

The first line reads as "in the rows where `a` is over 1, set `b` to zero",
and does nothing whatever.  The first bracket made a new temporary table,
the assignment changed that, and it was discarded — `toy` never saw it.
No error is raised, which is what makes it dangerous.

The second line does the selection and the assignment in one step, and
works.  The rule is short enough to memorise: **when you are changing
values, use `.loc`, with the rows and the column inside a single pair of
brackets.**



### I changed a value and nothing happened



-   **Ask:** I wrote `df[df["a"] > 1]["b"] = 0` and `df` is unchanged, with no
    error.
-   **Answer:** The first bracket produced a temporary copy and the assignment
    changed the copy, which was then thrown away.  Selecting from a selection
    gives you a copy, and assigning to a copy has no effect on the original.
    Write it as one step instead: `df.loc[df["a"] > 1, "b"] = 0`.  Whenever
    you are assigning rather than just looking, use `.loc`.



## 1.  Missing values



Real surveys have gaps.  Python writes a missing number as `NaN`, for "not
a number", and it behaves unlike anything else.



In [1]:
import numpy as np

print("nan > 5   ->", np.nan > 5)
print("nan == nan ->", np.nan == np.nan)
print("missing ages in 2016-17:", int(w7["Age"].isna().sum()))

Every comparison involving a missing value is false — including comparing
it with itself.  So a filter like `df["Age"] > 60` silently drops the rows
where age is unknown, and `df["Age"] =` np.nan= never finds anything at
all, however many gaps there are.

The habit to form: ask about missing explicitly with `.isna()`, and never
by comparison.  Here that reports none missing in 2016–17, which is worth
knowing rather than assuming.



## 1.  Summarising by group



To compute something for each group, use `groupby`.



In [1]:
print(flat.groupby(flat["t"].astype(str)).size().to_string())

One line per round, from 15,492 people in 1987–88 to 72,372 in 2012–13.
`groupby` splits the table by the value you name, `size()` counts each
piece, and — again — `flat` itself is untouched.

Python does have loops, and you will meet `for` in the notebooks, but for
work like this you rarely want one.  Operating on whole columns at a time is
shorter to write, faster to run, and handles missing values correctly.



### Should I write a loop over the rows?



-   **Ask:** How do I go through each household one at a time to compute
    something?
-   **Answer:** Usually you should not.  Whole-column operations like
    `df["Age"].mean()` and `groupby` do the same job, run far faster, and
    deal with missing values properly.  Loops earn their place when you are
    iterating over a handful of things — files, or survey rounds — rather
    than over rows, of which there are a quarter of a million here.



## Where to go next



That is enough to read the session notebooks and change them without
guessing.  Three things are worth keeping in front of you:

-   **Assignment:** use `.loc` whenever you are changing values, never a
    bracket after a bracket.
-   **The index:** `reset_index()` turns index levels into ordinary columns,
    which is usually what you want while learning.
-   **Missing:** test with `.isna()`, never with a comparison.

When something breaks, read the last line of the message first: it names
the kind of error and usually the label or value that caused it.  The
notebooks are set up to show you the failing line and little else.  If it is
still opaque, ask — errors in this workshop are data about the materials,
not evidence about you.

